In [9]:
!pip install librosa soundfile numpy matplotlib scipy > /dev/null

In [11]:
"""
phase_vs_mgdf_experiment.py

Experimentos comparativos: magnitude, phase (wrapped/unwrapped), group delay (GD) e MGDF.
Gera: figuras PNG e arquivos WAV (original, phase-only, mag-only, swapped-phase).

Dependências:
  pip install librosa soundfile numpy matplotlib scipy
"""
import os
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# ========== parâmetros ==========
SR = 22050                # sample rate alvo
N_FFT = 2048
HOP = 512
WIN_LEN = N_FFT
GAMMA = 0.2               # parâmetro MGDF (Murty & Yegnanarayana)
ALPHA = 0.4               # parâmetro MGDF
SMOOTH_SIGMA = 1.0        # suavização espectral (em bins) para magnitude antes de MGDF
EPS = 1e-8

# arquivos de entrada
FILE1 = "voice01.wav"
FILE2 = "oi_lindo.wav"

# checagens iniciais
if not os.path.exists(FILE1) or not os.path.exists(FILE2):
    raise FileNotFoundError(f"Coloque '{FILE1}' e '{FILE2}' no diretório atual.")

# ========== utilitários ==========
def load_mono(path, sr=SR):
    x, _ = librosa.load(path, sr=sr, mono=True)
    # normalizar amplitude para evitar saturação de escrita
    x = x / (np.max(np.abs(x)) + 1e-12)
    return x

def pad_or_trim(x1, x2):
    L = min(len(x1), len(x2))
    return x1[:L], x2[:L]

def stft(x):
    return librosa.stft(x, n_fft=N_FFT, hop_length=HOP, win_length=WIN_LEN, window='hann')

def istft(X):
    return librosa.istft(X, hop_length=HOP, win_length=WIN_LEN, window='hann', length=None)

# ========== operações de fase / GD / MGDF ==========
def unwrap_phase(X):
    # X: complex STFT (freq_bins x frames)
    return np.unwrap(np.angle(X), axis=0)  # unwrapping along frequency axis

def compute_group_delay_from_spectrum(X):
    """
    Compute group delay per frame using frequency-derivative formula:
    tau(omega) = - d phi / d omega = (Re(X) * dIm(X)/dω - Im(X) * dRe(X)/dω) / |X|^2
    We approximate derivative wrt frequency by finite differences across frequency bins.
    Returns array same shape as X (freq x frames).
    """
    real = np.real(X)
    imag = np.imag(X)
    # derivative along frequency bins (axis=0)
    dreal = np.gradient(real, axis=0)
    dimag = np.gradient(imag, axis=0)
    denom = real**2 + imag**2 + EPS
    numer = real * dimag - imag * dreal
    gd = numer / denom
    # optional: negative sign convention depending on definition; we'll keep this form
    return gd

def smooth_magnitude(mag, sigma=SMOOTH_SIGMA):
    # smooth magnitude across frequency bins per frame
    sm = gaussian_filter1d(mag, sigma=sigma, axis=0, mode='reflect')
    return sm

def compute_mgdf(X, gamma=GAMMA, alpha=ALPHA, sigma=SMOOTH_SIGMA):
    """
    Compute Modified Group Delay Function (MGDF) per Murty & Yegnanarayana.
    Steps (practical approximation):
      - magnitude smoothing
      - raise smoothed magnitude to power gamma, multiply STFT -> Y = X * (smoothed_mag**gamma)
      - compute numerator using derivative of Y (as in GD formula)
      - denominator uses (smoothed_mag)^(2*alpha)
    Returns MGDF matrix (freq x frames).
    """
    mag = np.abs(X)
    smag = smooth_magnitude(mag, sigma=sigma) + EPS
    weight = smag ** gamma
    Y = X * weight
    # derivatives of Y
    Y_real = np.real(Y)
    Y_imag = np.imag(Y)
    dY_real = np.gradient(Y_real, axis=0)
    dY_imag = np.gradient(Y_imag, axis=0)
    numer = Y_real * dY_imag - Y_imag * dY_real
    denom = (smag ** (2 * alpha)) + EPS
    mgdf = numer / denom
    return mgdf

# ========== carregar sinais ==========
x1 = load_mono(FILE1, sr=SR)
x2 = load_mono(FILE2, sr=SR)

# alinhar comprimentos
x1, x2 = pad_or_trim(x1, x2)

# aplicar leve preênfase (opcional, pode realçar excitação glotal)
pre_emph = 0.97
x1 = np.append(x1[0], x1[1:] - pre_emph * x1[:-1])
x2 = np.append(x2[0], x2[1:] - pre_emph * x2[:-1])

# ========== STFTs ==========
S1 = stft(x1)
S2 = stft(x2)
mag1 = np.abs(S1)
mag2 = np.abs(S2)
phase1 = np.angle(S1)
phase2 = np.angle(S2)
unwrapped1 = unwrap_phase(S1)
unwrapped2 = unwrap_phase(S2)

# ========== GD e MGDF ==========
GD1 = compute_group_delay_from_spectrum(S1)
GD2 = compute_group_delay_from_spectrum(S2)

MGDF1 = compute_mgdf(S1, gamma=GAMMA, alpha=ALPHA, sigma=SMOOTH_SIGMA)
MGDF2 = compute_mgdf(S2, gamma=GAMMA, alpha=ALPHA, sigma=SMOOTH_SIGMA)

# normalizar para plots (melhor visualização)
def norm01(A):
    A = np.array(A, dtype=float)
    A = A - np.nanmin(A)
    if np.nanmax(A) > 0:
        A = A / (np.nanmax(A) + 1e-12)
    return A

# ========== reconstruções áudio (experimentais) ==========
# 1) original (reconstruct to ensure consistency)
y1_orig = istft(S1)
y2_orig = istft(S2)
# 2) phase-only (unity magnitude + original phase)
unity_mag = np.ones_like(S1)
y1_phase_only = istft(unity_mag * np.exp(1j * phase1))
y2_phase_only = istft(unity_mag * np.exp(1j * phase2))
# 3) mag-only (original magnitude + zero phase)
y1_mag_only = istft(mag1 * np.exp(1j * np.zeros_like(phase1)))
# 4) swapped-phase (magnitude of 1 + phase of 2)
# ensure shapes match (they do because we aligned signals)
y_swap = istft(mag1 * np.exp(1j * phase2))

# salvar WAVs (normalizar p/ evitar clipping)
def save_wav(x, path, sr=SR):
    x = x / (np.max(np.abs(x)) + 1e-12) * 0.95
    sf.write(path, x, sr)

os.makedirs("outputs", exist_ok=True)
save_wav(y1_orig, "outputs/voz1_original.wav")
save_wav(y2_orig, "outputs/voz2_original.wav")
save_wav(y1_phase_only, "outputs/voz1_phase_only.wav")
save_wav(y2_phase_only, "outputs/voz2_phase_only.wav")
save_wav(y1_mag_only, "outputs/voz1_mag_only.wav")
save_wav(y_swap, "outputs/voz1_mag_with_voz2_phase.wav")

# ========== plots comparativos ==========
import matplotlib
matplotlib.use('Agg')  # headless safe

freqs = np.linspace(0, SR/2, N_FFT//2 + 1)
frame_idx = S1.shape[1] // 4  # escolher um quadro para plot 1D (terceira parte)

# 1) espectrogramas (magnitude)
plt.figure(figsize=(12, 6))
plt.subplot(2,2,1)
plt.title("Magnitude (voz1) - espectrograma (dB)")
librosa.display.specshow(librosa.amplitude_to_db(mag1, ref=np.max), sr=SR, hop_length=HOP, y_axis='linear', x_axis='time')
plt.colorbar(format='%+2.0f dB')

plt.subplot(2,2,2)
plt.title("Magnitude (voz2) - espectrograma (dB)")
librosa.display.specshow(librosa.amplitude_to_db(mag2, ref=np.max), sr=SR, hop_length=HOP, y_axis='linear', x_axis='time')
plt.colorbar(format='%+2.0f dB')

# 2) unwrapped phase slice vs
plt.subplot(2,2,3)
plt.title("Unwrapped phase (freq slice) - quadro idx {}".format(frame_idx))
plt.plot(freqs, unwrapped1[:, frame_idx], label='voz1')
plt.plot(freqs, unwrapped2[:, frame_idx], label='voz2', alpha=0.8)
plt.xlabel("Hz")
plt.legend()

# 3) MGDF images
plt.tight_layout()
plt.savefig("outputs/magnitude_and_phase_slice.png", dpi=200)

# plot GD and MGDF as images for visual comparação
plt.figure(figsize=(12,6))
plt.subplot(2,2,1)
plt.title("Group Delay (voz1) - image (normalized)")
plt.imshow(norm01(GD1), origin='lower', aspect='auto', extent=[0, x1.shape[0]/SR, 0, SR/2])
plt.colorbar()
plt.subplot(2,2,2)
plt.title("Group Delay (voz2) - image (normalized)")
plt.imshow(norm01(GD2), origin='lower', aspect='auto', extent=[0, x2.shape[0]/SR, 0, SR/2])
plt.colorbar()
plt.subplot(2,2,3)
plt.title("MGDF (voz1) - image (normalized)")
plt.imshow(norm01(MGDF1), origin='lower', aspect='auto', extent=[0, x1.shape[0]/SR, 0, SR/2])
plt.colorbar()
plt.subplot(2,2,4)
plt.title("MGDF (voz2) - image (normalized)")
plt.imshow(norm01(MGDF2), origin='lower', aspect='auto', extent=[0, x2.shape[0]/SR, 0, SR/2])
plt.colorbar()
plt.tight_layout()
plt.savefig("outputs/gd_mgdf_images.png", dpi=200)

# plot 1D slices to inspect differences at chosen frame
plt.figure(figsize=(10,6))
plt.subplot(2,1,1)
plt.title("MGDF - quadro idx {}".format(frame_idx))
plt.plot(freqs, norm01(MGDF1[:, frame_idx]), label='voz1 MGDF')
plt.plot(freqs, norm01(MGDF2[:, frame_idx]), label='voz2 MGDF', alpha=0.8)
plt.legend()
plt.subplot(2,1,2)
plt.title("Group Delay (same quadro)")
plt.plot(freqs, norm01(GD1[:, frame_idx]), label='voz1 GD')
plt.plot(freqs, norm01(GD2[:, frame_idx]), label='voz2 GD', alpha=0.8)
plt.legend()
plt.tight_layout()
plt.savefig("outputs/mgdf_gd_slices.png", dpi=200)

print("===== experiment complete =====")
print("WAVs salvos em: outputs/*.wav")
print("Figuras salvas em: outputs/*.png")


===== experiment complete =====
WAVs salvos em: outputs/*.wav
Figuras salvas em: outputs/*.png


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
mgdf_biometrics_analysis.py

Objetivo:
  - Extrair MGDF de sinais de voz
  - Extrair features agrupadas por bandas mel (redução dimensional)
  - Gerar comparações visuais: espectrograma, MGDF image, média por banda mel
  - Projetar features com PCA e plotar separabilidade
  - Treinar classificador simples (LogisticRegression) com validação cruzada
  - Plotar ROC e calcular AUC e EER

Uso:
  - Criar pastas `speakerA/` e `speakerB/` contendo WAVs (mesma frase idealmente).
  - Ou colocar apenas `voz1.wav` e `voz2.wav` (cada arquivo será segmentado).
  - Instalar dependências:
      pip install librosa soundfile numpy matplotlib scipy scikit-learn

Saídas:
  - pasta `outputs/` com PNGs, CSV com features e métricas no terminal.
"""
import os
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import accuracy_score
import csv
import warnings
warnings.filterwarnings("ignore")

# ========== parâmetros ==========
SR = 22050
N_FFT = 2048
HOP = 512
WIN_LEN = N_FFT
GAMMA = 0.2
ALPHA = 0.4
SMOOTH_SIGMA = 1.0
EPS = 1e-8
SEGMENT_SEC = 1.0    # duração do segmento para gerar múltiplos samples (se necessário)
MEL_BANDS = 40      # número de bandas mel para sumarização
RANDOM_STATE = 42

INPUT_A = "speakerA"
INPUT_B = "speakerB"
OUTDIR = "outputs_mgdf"
os.makedirs(OUTDIR, exist_ok=True)

# ========== utilitários ==========
def load_mono(path, sr=SR):
    x, _ = librosa.load(path, sr=sr, mono=True)
    x = x / (np.max(np.abs(x)) + 1e-12)
    return x

def segment_signal(x, sr=SR, seg_sec=SEGMENT_SEC):
    seg_len = int(seg_sec * sr)
    if len(x) <= seg_len:
        return [x]
    segments = []
    # non overlapping segments
    nseg = len(x) // seg_len
    for i in range(nseg):
        seg = x[i*seg_len:(i+1)*seg_len]
        segments.append(seg)
    # remaining tail (if > 0.5 seg) keep it
    tail = len(x) - nseg*seg_len
    if tail >= seg_len//2:
        segments.append(x[-seg_len:])
    return segments

def stft(x):
    return librosa.stft(x, n_fft=N_FFT, hop_length=HOP, win_length=WIN_LEN, window='hann')

def istft(X):
    return librosa.istft(X, hop_length=HOP, win_length=WIN_LEN, window='hann')

def smooth_magnitude(mag, sigma=SMOOTH_SIGMA):
    return gaussian_filter1d(mag, sigma=sigma, axis=0, mode='reflect')

def compute_mgdf(X, gamma=GAMMA, alpha=ALPHA, sigma=SMOOTH_SIGMA):
    mag = np.abs(X)
    smag = smooth_magnitude(mag, sigma=sigma) + EPS
    weight = smag ** gamma
    Y = X * weight
    Y_real = np.real(Y)
    Y_imag = np.imag(Y)
    dY_real = np.gradient(Y_real, axis=0)
    dY_imag = np.gradient(Y_imag, axis=0)
    numer = Y_real * dY_imag - Y_imag * dY_real
    denom = (smag ** (2 * alpha)) + EPS
    mgdf = numer / denom
    return mgdf

def mel_pooling_matrix(sr=SR, n_fft=N_FFT, n_mels=MEL_BANDS):
    # cria filtro mel (freq_bins x mel_bands) para agregar MGDF por banda
    mel_fb = librosa.filters.mel(sr=sr, n_fft=n_fft, n_mels=n_mels, fmin=0, fmax=sr/2)
    # mel_fb has shape (n_mels, 1 + n_fft/2). We want matrix (freq_bins, n_mels)
    return mel_fb.T  # shape freq_bins x n_mels

# ========== carregar arquivos e gerar segmentos ==========
def gather_segments_from_folder(folder):
    segs = []
    filenames = []
    if not os.path.isdir(folder):
        return segs, filenames
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.wav','.flac','.mp3','.ogg')):
            continue
        path = os.path.join(folder, fname)
        x = load_mono(path, sr=SR)
        segs_i = segment_signal(x, sr=SR, seg_sec=SEGMENT_SEC)
        for j, seg in enumerate(segs_i):
            segs.append(seg)
            filenames.append(f"{fname}::seg{j}")
    return segs, filenames

# allow two modes: folders or single files (voz1.wav voz2.wav)
def load_all_segments():
    X_segs = []
    y = []
    names = []
    # speaker A
    if os.path.isdir(INPUT_A):
        segsA, namesA = gather_segments_from_folder(INPUT_A)
    elif os.path.exists("voz1.wav"):
        segsA = segment_signal(load_mono("voz1.wav"), sr=SR, seg_sec=SEGMENT_SEC)
        namesA = [f"voz1_seg{i}" for i in range(len(segsA))]
    else:
        raise FileNotFoundError(f"Coloque pasta '{INPUT_A}' com WAVs, ou arquivo 'voz1.wav' no diretório.")
    # speaker B
    if os.path.isdir(INPUT_B):
        segsB, namesB = gather_segments_from_folder(INPUT_B)
    elif os.path.exists("voz2.wav"):
        segsB = segment_signal(load_mono("voz2.wav"), sr=SR, seg_sec=SEGMENT_SEC)
        namesB = [f"voz2_seg{i}" for i in range(len(segsB))]
    else:
        raise FileNotFoundError(f"Coloque pasta '{INPUT_B}' com WAVs, ou arquivo 'voz2.wav' no diretório.")

    if len(segsA) == 0 or len(segsB) == 0:
        raise ValueError("Cada falante precisa ter pelo menos 1 segmento válido (arquivo curto demais?).")

    for s, n in zip(segsA, namesA):
        X_segs.append(s)
        y.append(0)
        names.append(f"A::{n}")
    for s, n in zip(segsB, namesB):
        X_segs.append(s)
        y.append(1)
        names.append(f"B::{n}")

    return X_segs, np.array(y), names

# ========== pipeline principal ==========
print("Carregando segmentos...")
X_segs, y, seg_names = load_all_segments()
print(f"Segmentos carregados: {len(X_segs)} (A:{np.sum(y==0)}, B:{np.sum(y==1)})")

# compute mel pooling matrix
mel_mat = mel_pooling_matrix(sr=SR, n_fft=N_FFT, n_mels=MEL_BANDS)  # shape: freq_bins x n_mels
freq_bins = mel_mat.shape[0]

# extrair features MGDF por segmento
features = []
for idx, seg in enumerate(X_segs):
    # aplicar pre-emphasis
    pre_emph = 0.97
    segp = np.append(seg[0], seg[1:] - pre_emph * seg[:-1])
    S = stft(segp)
    mgdf = compute_mgdf(S, gamma=GAMMA, alpha=ALPHA, sigma=SMOOTH_SIGMA)  # shape freq x frames
    # agregação: primeiro média ao longo dos frames -> vector freq_bins
    mgdf_mean_freq = np.mean(mgdf, axis=1)
    # então pooling mel (freq_bins -> mel_bands)
    # shape mel_bands = mel_mat.T @ mgdf_mean_freq? mel_mat is freq_bins x mel_bands
    mel_feats = mel_mat.T.dot(mgdf_mean_freq)  # shape n_mels
    # padronizar energia / escala não necessária aqui, mas guardaremos
    features.append(mel_feats.real)  # garantir real
features = np.array(features)  # shape samples x n_mels

# normalizar e salvar features
scaler = StandardScaler().fit(features)
features_z = scaler.transform(features)

# salvar CSV de features
csv_out = os.path.join(OUTDIR, "features_mgdf_mel.csv")
with open(csv_out, "w", newline='') as f:
    writer = csv.writer(f)
    header = ["name", "label"] + [f"mel_{i}" for i in range(features.shape[1])]
    writer.writerow(header)
    for i, row in enumerate(features):
        writer.writerow([seg_names[i], int(y[i])] + row.tolist())
print(f"Features salvas em {csv_out}")

# ========== visualizações ==========

# 1) imagem MGDF média por falante (média sobre todos os segmentos de cada falante)
def plot_mgdf_images_per_speaker():
    # média por falante (freq x ?) -> vamos recomputar MGDF imagens médias ao longo do tempo para um segmento representativo
    # para ilustração, pegamos primeiro segmento de cada falante e plotamos MGDF spectro-image
    idxA = np.where(y==0)[0][0]
    idxB = np.where(y==1)[0][0]
    segA = X_segs[idxA]
    segB = X_segs[idxB]
    S_A = stft(np.append(segA[0], segA[1:] - 0.97*segA[:-1]))
    S_B = stft(np.append(segB[0], segB[1:] - 0.97*segB[:-1]))
    MGDF_A = compute_mgdf(S_A)
    MGDF_B = compute_mgdf(S_B)
    # normalizar para visual
    def norm01(A):
        A = A - np.min(A)
        if np.max(A) > 0:
            return A / (np.max(A) + 1e-12)
        return A
    plt.figure(figsize=(10,6))
    plt.subplot(2,1,1)
    plt.imshow(norm01(MGDF_A), origin='lower', aspect='auto', extent=[0, len(segA)/SR, 0, SR/2])
    plt.title("MGDF - exemplar speaker A")
    plt.colorbar()
    plt.subplot(2,1,2)
    plt.imshow(norm01(MGDF_B), origin='lower', aspect='auto', extent=[0, len(segB)/SR, 0, SR/2])
    plt.title("MGDF - exemplar speaker B")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, "mgdf_example_images.png"), dpi=200)
    plt.close()
    print("MGDF example images salvas.")

plot_mgdf_images_per_speaker()

# 2) média por banda mel (comparação A vs B)
mean_A = np.mean(features[y==0], axis=0)
mean_B = np.mean(features[y==1], axis=0)
plt.figure(figsize=(8,4))
plt.plot(mean_A, label='mean MGDF mel - A')
plt.plot(mean_B, label='mean MGDF mel - B', alpha=0.8)
plt.xlabel("Mel band index")
plt.title("MGDF média por banda mel (segmentos agregados)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "mgdf_mean_by_mel.png"), dpi=200)
plt.close()
print("Média por banda mel salva.")

# 3) PCA 2D scatter para visualizar separabilidade
pca = PCA(n_components=2, random_state=RANDOM_STATE)
proj = pca.fit_transform(features_z)
plt.figure(figsize=(7,6))
plt.scatter(proj[y==0,0], proj[y==0,1], label='A', alpha=0.7)
plt.scatter(proj[y==1,0], proj[y==1,1], label='B', alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA 2D - features MGDF (mel pooled)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "pca_mgdf_scatter.png"), dpi=200)
plt.close()
print("PCA scatter salvo.")

# ========== classificador + ROC / EER ==========
clf = LogisticRegression(random_state=RANDOM_STATE, solver='liblinear')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# obter probabilidades via cross-validated predictions
y_probs = cross_val_predict(clf, features_z, y, cv=cv, method='predict_proba')[:,1]
y_preds = (y_probs >= 0.5).astype(int)
acc = accuracy_score(y, y_preds)
print(f"Classificador (logistic) - accuracy (threshold 0.5, CV): {acc:.3f}")

# ROC e AUC
fpr, tpr, thresholds = roc_curve(y, y_probs)
roc_auc = auc(fpr, tpr)

# EER computation: point where FAR ~= FRR (FAR = FPR, FRR = 1 - TPR)
fnr = 1 - tpr
# find threshold where abs(fnr - fpr) minimal
eer_idx = np.nanargmin(np.abs(fnr - fpr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0

# plot ROC
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})')
plt.plot([0,1],[0,1],'k--')
plt.scatter([fpr[eer_idx]],[tpr[eer_idx]], c='red', label=f'EER≈{eer:.3f}')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC - MGDF features (mel-pooled)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "roc_mgdf.png"), dpi=200)
plt.close()
print(f"ROC salvo. AUC = {roc_auc:.4f}, EER ≈ {eer:.4f}")

# salvar resumo de métricas
with open(os.path.join(OUTDIR, "metrics_summary.txt"), "w") as f:
    f.write(f"Samples: {len(features)}\n")
    f.write(f"Speaker A segments: {np.sum(y==0)}\n")
    f.write(f"Speaker B segments: {np.sum(y==1)}\n")
    f.write(f"Classifier: LogisticRegression (5-fold CV)\n")
    f.write(f"Accuracy (CV, threshold 0.5): {acc:.4f}\n")
    f.write(f"AUC: {roc_auc:.4f}\n")
    f.write(f"EER approx: {eer:.4f}\n")
print("Resumo de métricas salvo.")

print("Tudo concluído. Veja a pasta:", OUTDIR)
